# Day 10 — Customer Churn Prediction API with FastAPI

## Objective
Convert a trained PyTorch neural network into a REST API using FastAPI.

## Technologies
- Python
- PyTorch
- FastAPI
- Pydantic
- Pytest
- HTTPX

## Architecture
Client → FastAPI → Input Validation → Preprocessing → PyTorch Model → JSON Response

## API Endpoints
- GET `/` — Application status
- POST `/predict` — Predict customer churn probability

## Expected Outcome
A working API that loads a previously trained neural network,
accepts customer information as JSON, and returns churn predictions.

The model uses synthetic customer data for educational purposes.

In [2]:
!pip -q install fastapi uvicorn httpx pytest

In [3]:
from pathlib import Path
import subprocess
import sys

repo = Path("/content/churn-repo")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/njalagam23/neural-networks-from-scratch.git",
            str(repo)
        ],
        check=True
    )

project = repo / "customer-churn-app"

sys.path.insert(0, str(project))

print("Project loaded:", project.exists())

Project loaded: True


In [4]:
required_files = [
    "app/model.py",
    "app/preprocessing.py",
    "app/predict.py",
    "artifacts/churn_model.pth",
    "artifacts/churn_scaler.joblib",
    "artifacts/model_config.json"
]

for filename in required_files:
    print(f"{filename}: {(project / filename).exists()}")

app/model.py: True
app/preprocessing.py: True
app/predict.py: True
artifacts/churn_model.pth: True
artifacts/churn_scaler.joblib: True
artifacts/model_config.json: True


In [5]:
%%writefile /content/churn-repo/customer-churn-app/api.py
from fastapi import FastAPI
from pydantic import BaseModel, Field
from app.predict import ChurnPredictor

app = FastAPI(
    title="Customer Churn Prediction API",
    version="1.0.0"
)

predictor = ChurnPredictor()


class CustomerInput(BaseModel):
    usage: float = Field(ge=0)
    tenure: int = Field(ge=0)
    tickets: int = Field(ge=0)


@app.get("/")
def home():
    return {"message": "Churn Prediction API is running"}


@app.post("/predict")
def predict_churn(customer: CustomerInput):
    return predictor.predict(
        usage=customer.usage,
        tenure=customer.tenure,
        tickets=customer.tickets
    )

Writing /content/churn-repo/customer-churn-app/api.py


In [6]:
from fastapi.testclient import TestClient
from api import app

client = TestClient(app)

health_response = client.get("/")

print("Health status:", health_response.status_code)
print("Health response:", health_response.json())

prediction_response = client.post(
    "/predict",
    json={
        "usage": 5,
        "tenure": 8,
        "tickets": 3
    }
)

print("Prediction status:", prediction_response.status_code)
print("Prediction:", prediction_response.json())

Health status: 200
Health response: {'message': 'Churn Prediction API is running'}
Prediction status: 200
Prediction: {'churn_probability': 0.7995, 'risk_level': 'High Churn Risk'}


In [7]:
invalid_requests = [
    {"usage": -5, "tenure": 8, "tickets": 3},
    {"usage": 5, "tickets": 3},
    {"usage": "unknown", "tenure": 8, "tickets": 3},
    {"usage": 5, "tenure": 8, "tickets": -1}
]

for customer in invalid_requests:
    response = client.post("/predict", json=customer)
    print(response.status_code, customer)

422 {'usage': -5, 'tenure': 8, 'tickets': 3}
422 {'usage': 5, 'tickets': 3}
422 {'usage': 'unknown', 'tenure': 8, 'tickets': 3}
422 {'usage': 5, 'tenure': 8, 'tickets': -1}


In [9]:
from pathlib import Path

repo = Path("/content/churn-repo")
project = repo / "customer-churn-app"

print("Project exists:", project.exists())

for file in repo.rglob("test_*.py"):
    print(file.relative_to(repo))

Project exists: True
Tests/test_prediction.py


In [10]:
import shutil

tests_folder = project / "tests"
tests_folder.mkdir(parents=True, exist_ok=True)

# Restore Day 9 tests if stored in the old location
old_test = repo / "Tests/test_prediction.py"
new_test = tests_folder / "test_prediction.py"

if old_test.exists() and not new_test.exists():
    shutil.copy2(old_test, new_test)

print("Available test files:")
for file in tests_folder.glob("test_*.py"):
    print(file.name)

Available test files:
test_prediction.py


In [11]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v"],
    cwd=str(project),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collecting ... collected 10 items

tests/test_prediction.py::test_prediction_probability PASSED             [ 10%]
tests/test_prediction.py::test_customer_risk[5-8-3-High Churn Risk] PASSED [ 20%]
tests/test_prediction.py::test_customer_risk[30-48-0-Low Churn Risk] PASSED [ 30%]
tests/test_prediction.py::test_customer_risk[12-15-2-High Churn Risk] PASSED [ 40%]
tests/test_prediction.py::test_invalid_customer_inputs[-5-8-3] PASSED    [ 50%]
tests/test_prediction.py::test_invalid_customer_inputs[unknown-8-3] PASSED [ 60%]
tests/test_prediction.py::test_invalid_customer_inputs[5--2-3] PASSED    [ 70%]
tests/test_prediction.py::test_invalid_customer_inputs[5-8--1] PASSED    [ 80%]
te

In [12]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")
test_file = project / "tests/test_api.py"

test_file.write_text('''
import pytest
from fastapi.testclient import TestClient
from api import app

client = TestClient(app)


def test_home_endpoint():
    response = client.get("/")
    assert response.status_code == 200


def test_valid_prediction():
    response = client.post(
        "/predict",
        json={"usage": 5, "tenure": 8, "tickets": 3}
    )

    assert response.status_code == 200
    assert 0 <= response.json()["churn_probability"] <= 1


@pytest.mark.parametrize("customer", [
    {"usage": -5, "tenure": 8, "tickets": 3},
    {"usage": 5, "tickets": 3},
    {"usage": "unknown", "tenure": 8, "tickets": 3},
    {"usage": 5, "tenure": 8, "tickets": -1}
])
def test_invalid_requests(customer):
    response = client.post("/predict", json=customer)
    assert response.status_code == 422
''')

print("API tests created:", test_file.exists())

API tests created: True


In [13]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v"],
    cwd=str(project),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collecting ... collected 16 items

tests/test_api.py::test_home_endpoint PASSED                             [  6%]
tests/test_api.py::test_valid_prediction PASSED                          [ 12%]
tests/test_api.py::test_invalid_requests[customer0] PASSED               [ 18%]
tests/test_api.py::test_invalid_requests[customer1] PASSED               [ 25%]
tests/test_api.py::test_invalid_requests[customer2] PASSED               [ 31%]
tests/test_api.py::test_invalid_requests[customer3] PASSED               [ 37%]
tests/test_prediction.py::test_prediction_probability PASSED             [ 43%]
tests/test_prediction.py::test_customer_risk[5-8-3-High Churn Risk] PASSED [ 50%]
tests/test_